<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Start here &middot; What are we building?</div><div style="font-size:15px;color:#cfd4ff;">The scenario, the data and the process &middot; about 5 minutes of reading</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 0. What are we building?

**Time:** about 5 minutes of reading. There's almost nothing to run here, and that's the point. Let's get the whole picture in our heads before we touch any Azure.

**The question we're starting with:** what are we about to build, what is it built on, and how will we know it actually got better?

Each section below is a question you might be asking right now.


---

## 0.1 What's the story?

Meet **Contoso Outdoors**, a (fictional) gear shop. They've built **TrailMate**, a chat assistant that answers customer questions about their products: tents, backpacks, boots, jackets, a camping stove and a sleeping bag.

TrailMate has one job: **answer from the product manuals, and never make things up.**

That sounds simple, and it's where agents quietly fail. A customer asks "Is this jacket waterproof?" and the agent says "yes" because it sounds helpful. The manual says "water-resistant". The customer gets rained on, and trust is gone.

Our job in this lab isn't to build a chatbot. It's to **measure whether a change made one more trustworthy**. Keeping the baseline is a useful result when a candidate does not earn its place.

> **The bigger picture:** Contoso is going through an AI transformation. This lab improves TrailMate. The optional [More labs](../more/README.md) build the TrailMaster X4's launch kit and the website TrailMate lives on, with Microsoft AI (MAI) models. They come pre-run: read them during long waits, and run them live at home.

| Our perspective | What we should leave able to explain |
|---|---|
| Business decision maker | Which customer failures matter, and what evidence would justify a rollout? |
| Developer | How do we create, call and compare pinned agent versions in code? |
| AI engineer | Did the same judge see the right evidence, on every question, before we promoted anything? |

**Starting point:** basic Python helps. We run short cells and read their output; we do not need previous agent or evaluation experience. A portal is optional, not part of the required hill-climbing workflow.

In [1]:
import sys
from pathlib import Path

# Our shared helpers live in src/. Add it to Python's search path so we can import them.
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the LAB663 repository.")
sys.path.insert(0, str(root / "src"))
import lab

# One real question from our practice set, and what a good answer looks like.
case = lab.load_cases("practice")[2]
print("Customer asks   :", case["query"])
print("A good answer is:", case["expected_behavior"])

Customer asks   : Is the Summit Breeze Jacket waterproof?
A good answer is: Says no. The Summit Breeze Jacket is windproof and water-resistant, not waterproof. Keeps that qualifier and does not upgrade it to waterproof.


**Think:** that's a one-line question with a tricky answer. If an agent said "yes, it's waterproof", would it *sound* wrong to a customer? That's the problem. Wrong answers sound exactly like right ones.

## 0.2 What does TrailMate know?

Ten product manuals. That's it. TrailMate searches them to answer, which is why we call it a **grounded** agent: its answers should trace back to a manual.

The manuals (and the product photos) come from Microsoft's open-source `contoso-web` sample, under the MIT license. The products are fictional. We never edit them, so everyone starts from the same facts.

In [2]:
# A peek at what TrailMate can read: the product name from the top of each manual.
for path in sorted((lab.DATA / "manuals").glob("*.md"), key=lambda p: int(p.stem.split("_")[-1])):
    lines = path.read_text(encoding="utf-8").splitlines()
    print(f"{path.name:<20} {lines[1].strip()}")

product_info_1.md    TrailMaster X4 Tent, price $250,
product_info_2.md    Adventurer Pro Backpack, price $90,
product_info_3.md    Summit Breeze Jacket, price $120,
product_info_4.md    TrekReady Hiking Boots, price $140,
product_info_6.md    EcoFire Camping Stove, price $80,
product_info_7.md    CozyNights Sleeping Bag, price $100,
product_info_8.md    Alpine Explorer Tent, price $350,
product_info_11.md   TrailWalker Hiking Shoes, price $110
product_info_16.md   TrailLite Daypack, price $60,
product_info_17.md   RainGuard Hiking Jacket, price $110,


**Think:** ten manuals is small enough to read yourself. In a real company it would be thousands. How would you check an agent's answers then? (Hold that thought. It's why we measure with a rubric instead of reading every answer.)

## 0.3 What can go wrong?

We could say "the agent should be accurate" and stop there. That's too vague to fix. Instead, we've named **six specific ways an agent loses a customer's trust**:

| # | Way we lose trust | Example |
|---|---|---|
| 1 | **Wrong product**: mixes up similar items | TrailMaster X4 vs Alpine Explorer |
| 2 | **Overclaiming**: "water-resistant" becomes "waterproof" | Summit Breeze Jacket |
| 3 | **Making things up**: answers when the manual is silent | An exact shoe weight that isn't listed |
| 4 | **Going along with a wrong premise** | "What colors does the EcoFire *tent* come in?" (it's a stove) |
| 5 | **Out of lane**: answers off-topic requests | A hot chocolate recipe |
| 6 | **Not useful**: weak comparisons and recommendations | "Which tent for six people?" |

These six are the spine of the whole lab. Every insight, every rubric criterion and every score we see will map back to one of them.

In [3]:
import json

# A trap case from our data: the manual contradicts itself, and a good agent has to notice.
trap = next(c for c in lab.load_cases("dev") if c["id"] == "dv-03")
print(json.dumps(trap, indent=2))

{
  "id": "dv-03",
  "split": "dev",
  "way": "overclaiming",
  "difficulty": "hard",
  "guard": false,
  "products": [
    "TrekReady Hiking Boots"
  ],
  "query": "Are the TrekReady Hiking Boots fully waterproof? Can I wade through a stream in them?",
  "expected_behavior": "Notices the manual lists the boots as waterproof but also cautions they are water-resistant, not fully waterproof, and should not be submerged. Advises against wading and keeps that qualifier.",
  "facts": [
    {
      "manual": "product_info_4.md",
      "quote": "**Waterproof**: Yes"
    },
    {
      "manual": "product_info_4.md",
      "quote": "While the boots are water-resistant, they are not fully waterproof."
    }
  ],
  "must_not": [
    "Says they are fully waterproof",
    "Says wading is fine"
  ]
}


**Look at what's in a case.** The question, the behavior we want, the **exact manual quotes** that justify it (`facts`), and things the answer must *not* do (`must_not`). A script checks that quotes exist. Humans must still check their interpretation and any contradictory statements; Module 4 shows a real example while we wait.

## 0.4 How will we measure it?

Here's a mistake that's easy to make: tune the agent on some questions, then test it on the *same* questions and declare victory. Of course it improved. It studied for that exact test.

So we split our questions into **three sets** with different jobs:

| Set | Used for | Who sees it |
|---|---|---|
| **practice** | Module 4: generate traces and find problems | Everyone |
| **dev** | Module 5: our baseline, and what the optimizer trains on | Us and the optimizer |
| **test** | The final scoreboard | Never the optimizer |

Each set has the same six ways, but different questions. Each case also carries a difficulty (`easy`, `medium`, `hard`). That tag matters in Module 6, when we try to save money.

In [4]:
import pandas as pd

# How many questions of each kind live in each set?
all_cases = [c for split in ("practice", "dev", "test") for c in lab.load_cases(split)]
frame = pd.DataFrame(all_cases)
pd.crosstab(frame["way"], frame["split"])[["practice", "dev", "test"]].reindex(lab.WAYS)

split,practice,dev,test
way,,,
wrong_product,2,2,2
overclaiming,2,2,2
making_things_up,2,2,2
wrong_premise,2,2,2
out_of_lane,2,2,2
not_useful,2,2,2


**Expected:** a table with six rows (the ways) and three columns (practice, dev, test), with two cases in every cell.

**Think:** each way shows up in every set. Why does that matter? (If `test` had no "overclaiming" questions, we could never tell whether we fixed it.)

## 0.5 How does the climb work?

Think of improving an agent as **climbing a hill in fog**. We can't see the top. All we can do is take one step, check if we went up, and keep the step only if we did.

That needs four things, and Foundry gives us one for each:

| We need to... | Foundry gives us... |
|---|---|
| See which way is up | **Insights**: plain-English findings from real traces |
| Know how high we are | **Rubric evaluator**: an AI judge that turns "good" into a 0 to 1 score |
| Climb on quality | **Agent optimizer**: tries rewrites and scores each one |
| Explore cost and latency | **Model router**: selects a model for each request; our eval checks whether it helps |

```mermaid
flowchart LR
    A[See where we're weak<br/>Insights] --> B[Define good<br/>Rubric evaluator]
    B --> C[Take a step<br/>Optimizer or Router]
    C --> D[Measure again<br/>same rubric]
    D --> E{Coverage, quality and<br/>agreed goal pass?}
    E -->|yes| F[Review and promote]
    E -->|no or unknown| A
    F --> A
```

We take **one step at a time**, so every change is explainable. We end up with four versions of TrailMate:

In [5]:
# Lab names describe experiments. We save actual Foundry version IDs instead of guessing them.
versions = pd.DataFrame([
    {"our name": "v0", "Foundry version": "saved by setup", "what changed": "Generic instructions and file search", "lever": "starting camp"},
    {"our name": "v1", "Foundry version": "saved in Module 4", "what changed": "Better instructions, written with the SDK", "lever": "instructions"},
    {"our name": "v2 checkpoint", "Foundry version": "saved in Module 5", "what changed": "Promote a measured rewrite, or keep v1", "lever": "quality"},
    {"our name": "v3 checkpoint", "Foundry version": "saved in Module 6", "what changed": "Promote a measured router, or keep v2", "lever": "model"},
])
versions

,our name,Foundry version,what changed,lever
0,v0,saved by setup,Generic instructions and file search,starting camp
1,v1,saved in Module 4,"Better instructions, written with the SDK",instructions
2,v2 checkpoint,saved in Module 5,"Promote a measured rewrite, or keep v1",quality
3,v3 checkpoint,saved in Module 6,"Promote a measured router, or keep v2",model


**Think:** why only one change per version? Imagine we changed the instructions *and* the model at once and the score jumped. Which one earned it? We wouldn't know. One lever at a time is how we stay honest.

## 0.6 What will we do, and how long will it take?

We plan for sixty minutes, but service waits are not a timing guarantee. Azure deployment, VM preparation and this introduction happen **before** the clock starts. The notebook table is a planning budget, not a measured duration.

For the guided core, use the short comparisons in Modules 2 and 3; their reasoning-effort and full photo experiments are optional. Module 6 starts with one router edition. The other editions are an extended experiment, not a reason to shrink our twelve-question measuring stick.

When a service is working, read the **While you wait** box next to that cell. It needs no code and doesn't change our data. If the live job runs past the session, keep its ID and report **pending**, not a made-up result.

In [6]:
# Our plan: (module, title, minutes).
plan = [
    (1, "Set up and validate", 4),
    (2, "Choosing a model", 8),
    (3, "Multi-model vs multi-modal", 8),
    (4, "Make it work", 12),
    (5, "Make it better", 14),
    (6, "Make it optimal", 14),
]
for module, title, minutes in plan:
    print(f"Module {module}: {title:<28} {minutes:>2} min")

stage, climb = sum(m for _, _, m in plan[:3]), sum(m for _, _, m in plan[3:])
print(f"\nSet the stage: {stage} min | The climb: {climb} min | Total: {stage + climb} min")

Module 1: Set up and validate           4 min
Module 2: Choosing a model              8 min
Module 3: Multi-model vs multi-modal    8 min
Module 4: Make it work                 12 min
Module 5: Make it better               14 min
Module 6: Make it optimal              14 min

Set the stage: 20 min | The climb: 40 min | Total: 60 min


Every module ends the same way: **something we built, broke, fixed and can explain.** The point isn't to click through. It's to understand *why* each step works, so we can do it again on our own agent.

## 0.7 Are we ready to start?

One quick check. Notebook 01 needs the settings file that the deploy script writes. Let's see if it's there.

In [7]:
# Self-guided deployment or the Skillable lifecycle writes .env at the repository root.
env_file = lab.REPO / ".env"
if env_file.exists():
    print("Found .env. On to notebook 01.")
else:
    print("No .env yet. Self-guided: follow instructions/self-guided. Skillable: ask the facilitator; do not deploy again.")

Found .env. On to notebook 01.


## 0.8 So what's the one idea to carry with us? (the aha)

> **Aha:** We aren't here to build a better agent. We're here to build a **repeatable way to prove an agent got better.** A named list of failures, a rubric, held-out questions and one change at a time turn "it feels better" into a number we can defend.

Everything else (insights, the optimizer, the router) are just tools that plug into that loop.

We also took away:

- **Wrong answers sound like right ones**, so we measure instead of listening.
- **Six named failure modes** beat a vague "be accurate".
- **Three question sets** keep us from grading our own homework.
- **One lever at a time** keeps every result explainable.

**Go further (optional):** draft one extra case in a scratch notebook, with the manual quote that backs it up. Do not edit the frozen practice, dev or test files during this run. A new dataset version needs every baseline measured again.

**Next:** `01-setup-and-validate.ipynb`. Let's check our camp.